# 00 · Reproduce Model B on CUAD (Milestone 0)

Runs `Ihteshamstar/qwen3-4b-cuad-extractor` with its native prompt on 10 of the 102 official
CUAD test contracts, at the model card's chunking (60k/8k chars) and at the 32k/4k chunking
planned for credit agreements. It reports detection F1 against the card's numbers, plus
loop/ladder stats, latency per chunk, and GPU minutes.

**Kaggle:** Accelerator **GPU T4 x2** (not P100), Internet **on**, Add-ons → Secrets: attach
`HF_TOKEN` (write access). "Save & Run All" runs it in the background.
**Colab:** T4 runtime, add `HF_TOKEN` under 🔑 Secrets with notebook access enabled.

Resumable: finished chunks are pulled from the private HF dataset repo and skipped.
All logic lives in `src/`; this notebook only wires it together.

In [ ]:
REPO_URL = "https://github.com/z26qin/credit-extractor.git"
BRANCH = "m0-reproduce"
INSTALL_VLLM = True      # False -> transformers only

In [ ]:
import os, subprocess, sys
ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else (
    "/content" if os.path.exists("/content") else os.getcwd())
REPO = os.path.join(ROOT, "credit-extractor")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["-r", "requirements.txt"], check=True)
if INSTALL_VLLM:
    r = subprocess.run(pip + ["-r", "requirements-gpu.txt"], capture_output=True, text=True)
    print("vllm install:", "ok" if r.returncode == 0 else "FAILED -> transformers fallback\n" + r.stderr[-2000:])
print(subprocess.run([sys.executable, "-c",
    "import importlib\nfor m in ['torch','transformers','vllm','huggingface_hub']:\n"
    "  try: print(m, importlib.import_module(m).__version__)\n"
    "  except Exception as e: print(m, 'n/a', type(e).__name__)"],
    capture_output=True, text=True).stdout)

In [ ]:
from src.platform import setup
env = setup()
assert env.data_repo, "HF_TOKEN missing: attach it as a secret (outputs must sync to the Hub)"
assert env.gpus, "no GPU visible: pick GPU T4 x2 (Kaggle) or a T4 runtime (Colab)"

In [ ]:
from src import reproduce
cfg = reproduce.load_cfg()
plan = reproduce.prepare(env, cfg)      # CUAD download, Hub pull, run dirs (fingerprinted)

In [ ]:
# primary greedy pass (vLLM TP=2 -> vLLM TP=1 -> transformers), then the
# repeat-penalty ladder (1.05 -> 1.15 -> 1.25, temperature 0) on failed chunks only
used = reproduce.run(env, plan, cfg)
used

In [ ]:
results = reproduce.report(env, plan, cfg)   # also writes report.md + results.json to the Hub

In [ ]:
import pandas as pd
for name, rd in plan["run_dirs"].items():
    df = pd.read_csv(rd / "latency.csv")
    print(name); display(df.sort_values(["chunk_id", "attempt"]))
print(f"session wall time: {env.session_minutes():.1f} min")